In [ ]:
from pathlib import Path
import shutil

import pandas as pd
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
REY_DIR = DATA_DIR / "REY_DATASET"
REY_ROI_ROT0_DIR = REY_DIR / "REY_roi_rot0"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"

YOLO_ROT_DIR = BASE_DIR / "yolo" / "rey_rotation"

CSV_LIMPIO = TABLES_DIR / "rey_traza_limpia.csv"

print("CSV limpio existe:", CSV_LIMPIO.exists())
print("REY_ROI_ROT0_DIR existe:", REY_ROI_ROT0_DIR.exists())
print("YOLO_ROT_DIR:", YOLO_ROT_DIR)

In [ ]:
df = pd.read_csv(CSV_LIMPIO)

print("Número de muestras:", len(df))
display(df.head())

print("\nDistribución de rotaciones:")
print(df["rot"].value_counts().sort_index())

In [ ]:
def rot_a_nombre_clase(rot):
    if rot == -90:
        return "rot_m90"
    elif rot == 0:
        return "rot_0"
    elif rot == 90:
        return "rot_90"
    elif rot == 180:
        return "rot_180"
    else:
        raise ValueError("Rotación no esperada: " + str(rot))

df["clase_yolo"] = df["rot"].apply(rot_a_nombre_clase)

display(df[["fname", "rot", "clase_yolo"]].head())

print("\nDistribución:")
print(df["clase_yolo"].value_counts())

In [ ]:
if YOLO_ROT_DIR.exists():
    shutil.rmtree(YOLO_ROT_DIR)

clases = ["rot_m90", "rot_0", "rot_90", "rot_180"]
splits = ["train", "val", "test"]

for split in splits:
    for clase in clases:
        carpeta = YOLO_ROT_DIR / split / clase
        carpeta.mkdir(parents=True, exist_ok=True)

print("Carpetas creadas.")

In [ ]:
RANDOM_STATE = 7

df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["clase_yolo"]
)

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=df_temp["clase_yolo"]
)

print("Train:", len(df_train))
print("Val:", len(df_val))
print("Test:", len(df_test))

print("\nTrain:")
print(df_train["clase_yolo"].value_counts())

print("\nVal:")
print(df_val["clase_yolo"].value_counts())

print("\nTest:")
print(df_test["clase_yolo"].value_counts())

In [ ]:
def copiar_split(df_split, split):
    for i, fila in df_split.iterrows():
        origen = REY_ROI_ROT0_DIR / fila["fname"]
        clase = fila["clase_yolo"]
        destino = YOLO_ROT_DIR / split / clase / fila["fname"]
        
        if origen.exists():
            shutil.copy2(origen, destino)
        else:
            print("No existe:", origen)

copiar_split(df_train, "train")
copiar_split(df_val, "val")
copiar_split(df_test, "test")

print("Imágenes copiadas.")

In [ ]:
for split in splits:
    print("\n", split.upper())
    
    for clase in clases:
        carpeta = YOLO_ROT_DIR / split / clase
        n = len(list(carpeta.glob("*.png"))) + len(list(carpeta.glob("*.jpg")))
        print(clase, ":", n)